# 04 — UniGaze fine-tuning (Stage A: frozen backbone, Stage B: partial fine-tuning)

Opening or running this notebook top-to-bottom **does not train**: the two training cells (§24) are guarded by
`RUN_STAGE_A = False` / `RUN_STAGE_B = False`. Cells marked **GPU** use the graphics card.

**Model**: UniGaze-B/16 (ViT-Base, 86.6 M parameters, gaze-pretrained on joint datasets) + a small coordinate head.
**Target**: normalised screen (x, y), top-left origin, linear output (no sigmoid → no edge saturation).
**Loss optimised**: Huber on residuals in screen-diagonal units (`src/losses.py`).
**Model selection**: participant-averaged mean Euclidean pixel error on validation participants P007 and P008
(`selection_metric: val_mean_euclid_px_macro`). Training loss is never used to select a model; the test set is
never loaded here.

### Experiment plan (controlled sequence — each result informs the next)
1. **A0** Stage A with defaults (MLP head 256, lr 3e-4, wd 0.01). Compare with baselines from notebook 03.
2. **A1** head type `linear` vs `mlp`; head width 128/256/512 only if A0 shows under- or over-fitting.
3. **B0** Stage B, last 4 blocks, backbone lr 1e-5, head initialised from the best Stage A head.
4. **B1** if B0 overfits (validation rises while training falls): 2 blocks, or backbone lr 5e-6, or wd 0.1.
   If B0 underfits (both errors flat and high): 6 blocks or backbone lr 2e-5.
5. **B2** augmentation on vs off — keep only if validation improves.
6. Stop when two consecutive changes fail to improve validation by more than ~2 %. Record every run;
   do not keep re-trying small variations (that overfits the two validation participants).

## 1. Environment verification

In [ ]:
# --- Workspace setup -----------------------------------------------------------
# WHAT: makes the `src` package importable and loads training_config.yaml.
# WHY : every notebook uses the same code and the same configuration.
# PATHS: resolved from this notebook's folder, never from the terminal's cwd.
import sys
from pathlib import Path

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
if not (WORKSPACE / "src" / "config.py").is_file():
    raise RuntimeError(f"Open this notebook from training_workspace/notebooks (cwd = {Path.cwd()})")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.config import load_config
cfg = load_config(WORKSPACE / "training_config.yaml")
cfg.paths.ensure_output_dirs()
W, H = cfg.screen_px
print("Workspace     :", cfg.paths.workspace)
print("Dataset root  :", cfg.paths.dataset_root, "(read-only)")
print("Screen        :", W, "x", H, "px")

## 2. Imports and dependency versions

In [ ]:
import json, math, time
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from src.utils import environment_report, select_device, seed_everything, new_run_dir, write_json
env = environment_report()
print(json.dumps({k: env.get(k) for k in ("python", "torch", "torch_cuda", "torchvision", "timm",
                                           "unigaze", "numpy", "cv2", "gpu", "gpu_capability")}, indent=1))

## 3. Configuration
All hyper-parameters come from `training_config.yaml`. To try a variant without editing the file, use
`cfg.with_overrides({...})` below; the full resolved config is saved with every run.

In [ ]:
OVERRIDES = {
    # "stage_a.head_lr": 1e-4,
    # "model.head.type": "linear",
    # "stage_b.unfreeze_last_blocks": 2,
}
cfg = cfg.with_overrides(OVERRIDES)
print(json.dumps({k: cfg[k] for k in ("model", "loss", "sampler", "stage_a", "stage_b", "augmentation")}, indent=1))
print("config fingerprint:", cfg.fingerprint())

## 4. CUDA and GPU verification
`select_device` checks that this PyTorch build contains kernels for the GPU's architecture (sm_120 on the
RTX 5060) and runs a real CUDA smoke test. It stops with a diagnostic instead of silently training on the CPU.

In [ ]:
seed_everything(cfg["reproducibility"]["seed"], cfg["reproducibility"]["deterministic"])
hw = cfg["hardware"]
dev = select_device(hw["device"], allow_cpu=hw["allow_cpu_training"], precision=hw["precision"])
print(dev)
from src.utils import suggest_batch_size
print("suggested batch sizes -> features:", suggest_batch_size(dev.total_vram_gb, "features"),
      "| fine-tune:", suggest_batch_size(dev.total_vram_gb, "finetune"))

## 5. Dataset loading (train + validation manifests and normalised caches)

In [ ]:
from src.pipeline import split_source
tr_df, tr_src, tr_geo = split_source(cfg, "train")
va_df, va_src, va_geo = split_source(cfg, "validation")
from src.dataset import effective_sample_report
print("train usable frames:", int(tr_src.ok.sum()), "/", len(tr_df))
print("validation usable frames:", int(va_src.ok.sum()), "/", len(va_df))
pd.concat([effective_sample_report(tr_df).assign(split="train"), effective_sample_report(va_df).assign(split="validation")])

## 6. Dataset integrity assertions (fail fast)

In [ ]:
from src.dataset import require_audit_pass, manifest_hash
audit = require_audit_pass(cfg)
assert tr_df["sample_id"].is_unique and va_df["sample_id"].is_unique
assert np.isfinite(tr_df[["target_x_norm", "target_y_norm"]].to_numpy()).all()
assert tr_df[["target_x_norm", "target_y_norm"]].stack().between(0, 1).all()
MANIFEST_HASHES = {"train": manifest_hash(tr_df), "validation": manifest_hash(va_df)}
print("audit:", audit["decision"], "| manifest hashes:", MANIFEST_HASHES)

## 7. Participant split verification

In [ ]:
assert set(tr_df.participant_id) == set(cfg.splits["train"])
assert set(va_df.participant_id) == set(cfg.splits["validation"])
assert not set(tr_df.participant_id) & set(va_df.participant_id)
assert not set(tr_df.trial_id) & set(va_df.trial_id)
assert not set(tr_df.raw_pixel_sha256) & set(va_df.raw_pixel_sha256)
print("train:", sorted(tr_df.participant_id.unique()), "| validation:", sorted(va_df.participant_id.unique()),
      "| test: sealed")

## 8. Image preprocessing
Inputs are the cached UniGaze-normalised faces (notebook 02), RGB uint8 → ImageNet mean/std on the GPU
(the normalisation UniGaze was trained with: `wrap_transforms('basic_imagenet')`). Validation uses exactly the
same deterministic path. Augmentation (photometric only) applies to training batches **only if enabled**.

In [ ]:
print("input mode:", cfg["preprocessing"]["input_mode"], "| version:", cfg["preprocessing"]["version"])
print("augmentation:", cfg["augmentation"])
from src.preprocessing import IMAGENET_MEAN, IMAGENET_STD
print("ImageNet mean/std:", IMAGENET_MEAN, IMAGENET_STD)

## 9. Batch visualisation (balanced sampler: every participant equal, ≤ k frames per trial)

In [ ]:
from src.dataset import GazeFrameDataset, ParticipantTrialBalancedSampler, collate
sampler = ParticipantTrialBalancedSampler(tr_df, tr_src.ok, cfg["sampler"]["trials_per_participant_per_epoch"],
                                          cfg["sampler"]["frames_per_trial_per_epoch"], seed=cfg["reproducibility"]["seed"])
epoch_idx = list(iter(sampler))
print("frames per epoch:", len(epoch_idx))
print(tr_df.iloc[epoch_idx].groupby("participant_id").size().rename("frames this epoch").to_frame().T)
ds = GazeFrameDataset(tr_df, tr_src)
b = collate([ds[i] for i in epoch_idx[:12]])
fig, axes = plt.subplots(1, 12, figsize=(18, 2))
for a, im, t in zip(axes, b["image"], b["target"]):
    a.imshow(im.permute(1, 2, 0)); a.axis("off"); a.set_title(f"{t[0]*W:.0f},{t[1]*H:.0f}", fontsize=7)
plt.show()

## 10. Model loading (official UniGaze implementation, pretrained gaze checkpoint)

In [ ]:
from src.models import load_unigaze, build_regressor
unigaze, ckpt_info = load_unigaze(cfg, allow_download=False)
model = build_regressor(unigaze, cfg)
print(json.dumps(ckpt_info, indent=1))

## 11. Checkpoint verification

In [ ]:
assert ckpt_info["parameters"] == 86_569_194
model.to(dev.device).eval()
idx = np.where(va_src.ok)[0][:8]
from src.preprocessing import to_model_tensor
with torch.no_grad():
    out = model(to_model_tensor(np.stack([va_src(i) for i in idx])).to(dev.device))
print({k: tuple(v.shape) for k, v in out.items()})
print("UniGaze pitch/yaw (deg):", np.degrees(out["gaze_pitchyaw"].float().cpu().numpy()).round(1)[:4])
print("untrained head predicts ~screen centre:", out["screen_xy"].float().cpu().numpy().round(3)[:4])

## 12. Backbone freezing
Stage A freezes **every** UniGaze parameter (patch/pos embedding, CLS token, 12 blocks, final norm and the
pretrained `gaze_fc`). Stage B later unfreezes only the last N blocks and the final LayerNorm.
`model.train_mode()` keeps frozen blocks in eval mode so stochastic depth only acts on trained blocks.

In [ ]:
from src.models import freeze_all_backbone, unfreeze_last_blocks, count_parameters
freeze_all_backbone(model)
print("Stage A:", count_parameters(model))

## 13. Regression head
`LayerNorm → Linear(770→256) → GELU → Dropout → Linear(256→2)`; input = 768-d CLS feature + UniGaze (pitch, yaw).
Last layer initialised to output the screen centre. Linear output: predictions may leave [0, 1] during training
(that is what lets the model learn edge targets); clamping happens only at deployment.

In [ ]:
print(model.head)

## 14. Trainable parameter counts

In [ ]:
for n in (0, 2, 4):
    unfreeze_last_blocks(model, n)
    c = count_parameters(model)
    print(f"last {n} blocks unfrozen: trainable {c['trainable']:>11,} / {c['total']:,} ({100*c['trainable']/c['total']:.2f}%)")
freeze_all_backbone(model)

## 15. Loss function

In [ ]:
from src.losses import ScreenCoordinateLoss
loss_fn = ScreenCoordinateLoss(W, H, cfg["loss"]["type"], cfg["loss"]["huber_delta"])
t = torch.tensor([[0.5, 0.5]])
for px in (10, 60, 300):
    p = t + torch.tensor([[px / W, 0.0]])
    print(f"{px:4d}px error -> loss {loss_fn(p, t).item():.5f}")
print("Huber delta =", cfg["loss"]["huber_delta"], "diagonal =", round(cfg["loss"]["huber_delta"] * cfg.screen_diagonal_px), "px")

## 16. Optimizer (AdamW, differential learning rates)
Head and backbone get different learning rates; inside the unfrozen blocks the LR decays toward shallower blocks
(`layer_lr_decay`). Biases, norms and embeddings get no weight decay.

In [ ]:
from src.models import param_groups
unfreeze_last_blocks(model, cfg["stage_b"]["unfreeze_last_blocks"])
groups = param_groups(model, cfg["stage_b"]["head_lr"], cfg["stage_b"]["backbone_lr"],
                      cfg["stage_b"]["weight_decay"], cfg["stage_b"]["layer_lr_decay"])
print(pd.DataFrame([{"lr": g["lr"], "weight_decay": g["weight_decay"], "tensors": len(g["params"]),
                     "parameters": sum(p.numel() for p in g["params"]), "example": g["names"][0]} for g in groups]))
freeze_all_backbone(model)

## 17. Learning-rate scheduler (preview only — no training)
Linear warm-up then cosine decay to 1 %, stepped every optimizer step. `plateau` (halve on validation plateau)
and `onecycle` (UniGaze's upstream choice) are available in the config.

In [ ]:
from src.trainer import build_scheduler
dummy = torch.optim.AdamW([torch.nn.Parameter(torch.zeros(1))], lr=cfg["stage_a"]["head_lr"])
steps_per_epoch = math.ceil(len(epoch_idx) / cfg["stage_a"]["batch_size"])
sched, unit = build_scheduler(dummy, cfg["stage_a"]["scheduler"], steps_per_epoch * cfg["stage_a"]["epochs"],
                              steps_per_epoch * cfg["stage_a"]["warmup_epochs"])
lrs = []
for _ in range(steps_per_epoch * cfg["stage_a"]["epochs"]):
    lrs.append(dummy.param_groups[0]["lr"]); dummy.step(); sched.step()
plt.figure(figsize=(7, 3)); plt.plot(lrs); plt.xlabel("optimizer step"); plt.ylabel("learning rate")
plt.title("Stage A schedule preview"); plt.show()

## 18. Training epoch function
`src/trainer.py` → `finetune()` (Stage B) and `train_head_on_features()` (Stage A). Per step: forward under
autocast → loss in fp32 → scaled backward → (every `grad_accum_steps`) unscale → gradient-norm log → clip →
optimizer step → scheduler step. The source is shown below so you can read it here.

In [ ]:
import inspect
from src import trainer
print(inspect.getsource(trainer.finetune)[:3500], "...")

## 19. Validation epoch function

In [ ]:
print(inspect.getsource(trainer.predict_loader))
print(inspect.getsource(__import__("src.metrics", fromlist=["x"]).macro_mean_euclid_px))

## 20. Mixed-precision configuration
BF16 autocast when the GPU supports it (Blackwell does; no loss scaling needed), otherwise FP16 + GradScaler.
The loss and metrics are always computed in fp32.

In [ ]:
print("precision:", dev.precision, "| GradScaler active:", dev.precision == "fp16")

## 21. Early stopping

In [ ]:
print("Stage A patience:", cfg["stage_a"]["early_stopping_patience"], "epochs | Stage B:",
      cfg["stage_b"]["early_stopping_patience"], "epochs (validation checks without improvement)")
print("After stopping, the BEST checkpoint (not the last epoch) is restored.")

## 22. Checkpoint management
Each run gets a fresh folder `experiments/<name>_<NNN>/` (never reused). `best.pt` is replaced atomically only
when validation improves; `last.pt` holds model + optimizer + scheduler + scaler + early-stopping + RNG states for
resuming an interrupted run.

## 23. Training history logging

In [ ]:
print("history.csv  : per epoch - train/val loss, train/val error, lr, grad norm, epoch time, GPU memory")
print("step_log.csv : per optimizer step - loss, grad norm, learning rate of every parameter group")
print("run_info.json: checkpoint hash, manifest hashes, seeds, trainable parameters, best epoch/metric")

## 24. Training execution — **GPU, opt-in**

### Stage A: frozen backbone (head trained on cached features)
Requires the features from notebook 03 (they are extracted here automatically if missing).
Typical time: features 3–8 min once, then ~1 min of head training.

In [ ]:
RUN_STAGE_A = False                 # <- set True deliberately
from src.pipeline import extract_split_features, head_inputs
from src.trainer import train_head_on_features
INFO = {"unigaze_checkpoint": ckpt_info, "manifest_hashes": MANIFEST_HASHES,
        "config_fingerprint": cfg.fingerprint(), "seed": cfg["reproducibility"]["seed"], "device": dev.__dict__}
stage_a = None
if RUN_STAGE_A:
    ftr = extract_split_features(cfg, unigaze, ckpt_info["sha256"], "train", dev.device, dev.precision)
    fva = extract_split_features(cfg, unigaze, ckpt_info["sha256"], "validation", dev.device, dev.precision)
    use = cfg["model"]["head"]["use_gaze_angles"]
    tu, vu = np.asarray(ftr["usable"]), np.asarray(fva["usable"])
    freeze_all_backbone(model)
    run_a = new_run_dir(cfg.paths.experiments, "unigaze_b16_frozen_head")
    stage_a = train_head_on_features(cfg, model.head, head_inputs(ftr, use)[tu], tr_df[tu].reset_index(drop=True),
                                     head_inputs(fva, use)[vu], va_df[vu].reset_index(drop=True),
                                     run_a, dev.device, "stage_a", INFO)
    print({k: v for k, v in stage_a.items() if k != "history"})

### Stage B: last N blocks unfrozen — longer (≈ 3–10 min per epoch on a laptop RTX 5060)
The head starts from the best Stage A head (`INIT_HEAD_FROM`), so fine-tuning does not begin with a random head
that would push large, noisy gradients into the pretrained blocks. If you get *out of memory*, lower
`stage_b.batch_size` and raise `stage_b.grad_accum_steps` (same effective batch). To resume an interrupted run,
set `RESUME_FROM` to its `last.pt`.

In [ ]:
RUN_STAGE_B = False                 # <- set True deliberately
INIT_HEAD_FROM = None               # e.g. cfg.paths.experiments / "unigaze_b16_frozen_head_001" / "best.pt"
RESUME_FROM = None                  # e.g. cfg.paths.experiments / "unigaze_b16_partial_ft_001" / "last.pt"
from src.pipeline import stage_b_loaders
from src.trainer import finetune
stage_b = None
if RUN_STAGE_B:
    if INIT_HEAD_FROM is None and stage_a is not None:
        INIT_HEAD_FROM = Path(stage_a["run_dir"]) / "best.pt"
    if INIT_HEAD_FROM is not None:
        model.head.load_state_dict(torch.load(INIT_HEAD_FROM, weights_only=False)["head_state"])
    unfreeze_last_blocks(model, cfg["stage_b"]["unfreeze_last_blocks"])
    loader_fn, val_loader, va_full, _ = stage_b_loaders(cfg, dev.device)
    run_b = Path(RESUME_FROM).parent if RESUME_FROM else new_run_dir(cfg.paths.experiments, "unigaze_b16_partial_ft")
    stage_b = finetune(cfg, model, loader_fn, val_loader, va_full, run_b, dev.device, dev.precision, "stage_b",
                       {**INFO, "init_head_from": str(INIT_HEAD_FROM)}, resume_from=RESUME_FROM)
    print({k: v for k, v in stage_b.items() if k != "history"})

## 25. Best-model selection (across all runs so far)
Reads every `experiments/*/run_info.json`. The selection metric is validation only. Note the number of runs:
the more configurations you try, the more optimistic the best validation number becomes.

In [ ]:
runs = []
for info_path in sorted(cfg.paths.experiments.glob("*/run_info.json")):
    i = json.loads(info_path.read_text())
    if "best_val_metric" in i and not info_path.parent.name.startswith("datacurve_"):
        runs.append({"run": info_path.parent.name, "stage": i.get("stage"), "best_epoch": i.get("best_epoch"),
                     "epochs_run": i.get("epochs_run"), "val_mean_px_macro": i["best_val_metric"],
                     "trainable": i.get("trainable_parameters"), "blocks": i.get("unfrozen_blocks")})
runs_df = pd.DataFrame(runs).sort_values("val_mean_px_macro") if runs else pd.DataFrame()
print(len(runs_df), "completed runs"); runs_df

## 26. Learning curves
**How to read them**
* *Underfitting*: training and validation errors both high and close together, still falling at the end → train
  longer, raise LR a little, or unfreeze more blocks.
* *Overfitting*: training error keeps falling while validation flattens or rises (gap grows) → the best epoch is
  early; use fewer unfrozen blocks, lower backbone LR, more weight decay, or augmentation.
* *LR too high*: loss spikes, very large or oscillating gradient norm, validation jumps → halve the LR.
* *LR too low*: almost flat curves and tiny gradient norms → raise LR (×2–3).

In [ ]:
from src.visualization import plot_learning_curves, plot_step_log
from IPython.display import Image, display
RUN_TO_PLOT = runs_df.iloc[0]["run"] if len(runs_df) else None
if RUN_TO_PLOT:
    rd = cfg.paths.experiments / RUN_TO_PLOT
    figs = {**plot_learning_curves(pd.read_csv(rd / "history.csv"), rd / "figures", RUN_TO_PLOT),
            **plot_step_log(pd.read_csv(rd / "step_log.csv"), rd / "figures")}
    for p in figs.values():
        display(Image(filename=str(p)))

## 27. Prediction generation (all usable validation frames, best checkpoint) — GPU, opt-in

In [ ]:
RUN_VAL_PREDICTION = False          # <- set True (GPU, ~1-3 min for 10,000 frames)
from src.pipeline import predict_split
from src.evaluation import prediction_table
from src.models import load_unigaze, build_regressor
SELECTED_RUN = RUN_TO_PLOT           # or set a run name explicitly, e.g. "unigaze_b16_frozen_head_001"
val_pred = None
if SELECTED_RUN and RUN_VAL_PREDICTION:
    rd = cfg.paths.experiments / SELECTED_RUN
    best = torch.load(rd / "best.pt", map_location="cpu", weights_only=False)
    # Rebuild from a FRESH UniGaze backbone: a Stage B run earlier in this kernel changes the backbone in
    # place, and a Stage A checkpoint stores only the head.
    unigaze, ckpt_info = load_unigaze(cfg, allow_download=False)
    model = build_regressor(unigaze, cfg)
    if "model_state" in best:
        model.load_state_dict(best["model_state"])
    else:
        model.head.load_state_dict(best["head_state"])
    model.to(dev.device).eval()
    df_ok, xy, gaze, track = predict_split(cfg, model, "validation", dev.device, dev.precision)
    val_pred = prediction_table(df_ok, xy, W, H, f"{SELECTED_RUN}/best.pt", cfg["preprocessing"]["version"], SELECTED_RUN)
    val_pred.to_parquet(rd / "val_predictions_all_frames.parquet", index=False)
    print(track, "| mean error px:", round(val_pred["euclid_px"].mean(), 1))

## 28. Model evaluation (validation)

In [ ]:
from src.evaluation import evaluate_predictions
if val_pred is not None:
    val_eval = evaluate_predictions(val_pred, cfg, bootstrap=True)
    write_json(rd / "val_evaluation.json", val_eval)
    print(json.dumps({k: val_eval[k] for k in ("frame_level", "trial_level", "participant_macro_mean_euclid_px",
                                               "bootstrap_mean_euclid", "target_selection_success")}, indent=1))

## 29. Error analysis

In [ ]:
from src.visualization import plot_error_analysis, plot_prediction_map, plot_error_heatmap
if val_pred is not None:
    figs = plot_error_analysis(val_pred, W, H, rd / "figures", "validation", manifest=va_df)
    figs["map"] = plot_prediction_map(val_pred, W, H, rd / "figures" / "validation_prediction_map",
                                      f"Validation predictions - {SELECTED_RUN}")
    figs["heat"] = plot_error_heatmap(val_pred, W, H, rd / "figures" / "validation_error_heatmap")
    for p in figs.values():
        display(Image(filename=str(p)))

## 30. Model export
Writes `checkpoints/selected/gaze_model.pt` (backbone + head + preprocessing settings + selection record) for
notebooks 06/07 and the live application. Also writes `reports/model_selection_decision.json` — the decision made
**before** the test set is opened. Export only once you are finished experimenting.

In [ ]:
EXPORT = False                       # <- set True when the model is final
from src.infer import export_model
if EXPORT and val_pred is not None:
    selection = {"run": SELECTED_RUN, "selection_metric": cfg["selection_metric"],
                 "val_mean_euclid_px_macro": float(val_eval["participant_macro_mean_euclid_px"]),
                 "runs_compared": int(len(runs_df)), "decided": pd.Timestamp.now().isoformat()}
    out = export_model(model.cpu(), cfg, cfg.paths.checkpoints / "selected" / "gaze_model.pt", selection, ckpt_info)
    write_json(cfg.paths.reports / "model_selection_decision.json", selection)
    print("exported:", out)